In [ ]:
import os
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reproduce.py").is_file())
os.chdir(ROOT)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
 
PROJECT_PATH = Path("data/genomics") 

In [ ]:
meta = pd.read_csv(PROJECT_PATH / "reference/metadata_complete.csv")
meta['population'] = meta['Strain'] + '_' + meta['Culture'].astype(str).str.zfill(2)
meta

In [ ]:
meta_subset = meta.query("Strain == 'P' and Culture < 5")

dfs = []
for i, row in meta_subset.iterrows():
    df = pd.read_csv(PROJECT_PATH / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
    df["Strain"] = row['Strain']
    df["Culture"] = row['Culture']
    df["Day"] = int(row['Day'])
    dfs.append(df)

df = pd.concat(dfs)
df



In [ ]:
print(df.shape)
print(df['Culture'].value_counts())
print(df['Day'].unique())

In [ ]:
meta_subset = meta.query("Strain == 'P' and Culture < 5").copy()

# For P_01 Day 0, keep only the 20230901 version
mask = (meta_subset['population'] == 'P_01') & (meta_subset['Day'] == 0)
meta_subset = meta_subset[~mask | (meta_subset['FolderDate'] == '20230901')]

# Step 2: Get unique populations and timepoints for just those cultures
pops = meta_subset['population'].unique()
timepoints = sorted(meta_subset['Day'].unique())

print("Selected pops:", pops)
print("Timepoints:", timepoints)


In [ ]:
def traceAlleleFreq(pop, min_freq=0.33):
    print(f"[TRACE] Processing population: {pop} with min_freq={min_freq}")

    D = []
    sorted_meta = meta_subset.query(f'population=="{pop}"').sort_values(by='Day')
    for i, row in sorted_meta.iterrows():
        df = pd.read_csv(PROJECT_PATH / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
        D.append(df)

    # Remove mutations detected in the wild-type background
    # 1. Select background mutations with strong signal
    wt_background=D[0].loc[D[0].frequency>0.01, 'position']
    #print(f"WT background mutations: {wt_background.values}")
    D_=[]

    #print(f"# of mutations\t# of (freq>{min_freq}):")
    for i in range(len(D)):
        # 2. Filter out mutations by position on the chromosome
        df=D[i][~D[i]['position'].isin(wt_background)]
        D_.append(df)
        #print(f"{df.shape[0]}\t{sum(df['frequency']>min_freq)}")
    
    tracked_positions=[]
    for i in range(len(D_)):
        d=D_[i]
        tracked_positions.extend(list(d.loc[ (d['frequency']>min_freq), 'position' ])) #& ~d['mutation_category'].isin(['mobile_element_insertion','large_deletion']), 'position' ] ))
    tracked_positions=set(tracked_positions)
    print(f"Population {pop} # of tracked mutations: {len(tracked_positions)}")

    ## 3. Trace the frequency of those mutations
    T=[]
    for pos in tracked_positions:

        freq=[]
        for i in range(len(D_)):
            d = D_[i]
            ind = d['position']==pos
            
            if sum(ind)<1:
                freq.append(0)

            else:
                freq.append(d.loc[ind, 'frequency'].values[0])
                gene_name = d.loc[ind, 'gene_name'].values[0] if 'gene_name' in d else ''
                gene_product = d.loc[ind, 'gene_product'].values[0] if 'gene_product' in d else ''
                aa_ref_seq = d.loc[ind, 'aa_ref_seq'].values[0] if 'aa_ref_seq' in d else ''
                aa_new_seq = d.loc[ind, 'aa_new_seq'].values[0] if 'aa_new_seq' in d else ''
                new_seq = d.loc[ind, 'new_seq'].values[0] if 'new_seq' in d else ''
                codon_ref_seq = d.loc[ind, 'codon_ref_seq'].values[0] if 'codon_ref_seq' in d else ''
                aa_pos = d.loc[ind, 'aa_position'].values[0] if 'aa_position' in d else ''
                gene_pos = d.loc[ind, 'gene_position'].values[0] if 'gene_position' in d else ''
                mut_cat = d.loc[ind, 'mutation_category'].values[0] if 'mutation_category' in d else ''

        T.append({'position': pos, 'freq': np.round(freq,2), 
                'gene_name':gene_name, 'gene_product':gene_product,
                'aa_ref_seq':aa_ref_seq, 'aa_new_seq':aa_new_seq,
                'aa_pos': aa_pos, 'gene_pos':gene_pos, 'mut_cat': mut_cat,
                'new_seq': new_seq, 'codon_ref_seq': codon_ref_seq})

    T=pd.DataFrame(T)
    if T.empty:
        print(f"[INFO] Population {pop}: no mutations passed filtering — returning empty DataFrame.")
        return T

    T.fillna({'aa_ref_seq': '',
              'aa_new_seq': '',
              'aa_pos': ''},inplace=True)
    
    nsi = T['aa_pos']==''
    T.loc[nsi,'label'] = T.loc[nsi,'gene_name'] + ' ' + T.loc[nsi, 'mut_cat']
    T.loc[~nsi, 'label'] = T.loc[~nsi,'gene_name'] + ' ' + T.loc[~nsi,'aa_ref_seq'] + T.loc[~nsi, 'aa_pos'].astype(str).str.replace(r'\.0','',regex=True) + T.loc[~nsi,'aa_new_seq']
        # T.loc[nsi,'gene_pos'] + ' '
    T.sort_values(by=['mut_cat','gene_name'],ascending=False,inplace=True)
    T.reset_index(inplace=True,drop=True)
    
    return T

In [ ]:
af=[]
for pop in pops:
    # print(traceAlleleFreq(pop, min_freq=0.2).shape)
    af.append(traceAlleleFreq(pop, min_freq=0.1))
    #af[-1].to_csv(f"data/genomics/data/processed/traced_alleles/{select_lineage}/{pop}.csv", index=False)

In [ ]:
pd.set_option("display.max_rows", 100)
af[0].head(100)

In [ ]:
for i, df in enumerate(af):
    if not df.empty:
        freq_lengths = df['freq'].apply(len).value_counts().sort_index()
        print(f"Culture {i+1} (Pop: {pops[i]}):")
        for length, count in freq_lengths.items():
            print(f"  {count} freq list(s) with length {length}")
        print("-" * 40)
    else:
        print(f"Culture {i+1} (Pop: {pops[i]}): EMPTY DataFrame")
        print("-" * 40)


In [ ]:
all = []
for ix, df in enumerate(af):
    dff = df.copy()
    dff['Pop'] = ix+1 
    all.append(dff)
combined_df=pd.concat(all)


combined_df['last_freq'] = combined_df['freq'].apply(lambda x: x[-1])
combined_df


In [ ]:
unique_labels = combined_df['label'].unique().tolist()
unique_labels

In [ ]:
def get_mutation_group_and_labels(group):
    # """
    # Returns select mutations and alternate labels based on the specified group.
    
    # Parameters:
    #     group (str): The group to retrieve mutations for. Options are 'prs_phoQ' or 'other'.
    
    # Returns:
    #     tuple: A tuple containing a list of selected mutations and a dictionary of alternate labels.
    # """
    # Select mutations based on group
        # Select mutations based on group
    if group == 'untreated':
        select_mutations = [
            'ygfB P184Q', 'yfbS V35F', 'treB V113E', 'fliI E290D', 'eutH P393Q',
            'lrhA/alaA snp_intergenic', 'lon small_indel', 'hns/tdk mobile_element_insertion',
            'fimE/fimA mobile_element_insertion', 'dsrA mobile_element_insertion', 'narQ L21L',
            'rpoS I128N', 'pitA A170V', 'obgE A74T', 'arcA M39I', 'yohP/dusC snp_intergenic',
            'yhaC/rnpB snp_intergenic', 'lysO/aqpZ snp_intergenic', 'rpoS small_indel',
            'rcsB mobile_element_insertion', 'pitA T184T', 'yaaA I52T', 'yhfG/ppiA snp_intergenic',
            'rlmB/yjfI snp_intergenic', 'nrfG/gltP snp_intergenic', 'manZ/yobD snp_intergenic',
            'lon E4E', 'rpoS L125R', 'insB9–[crl] large_deletion'
        ]

        alt_labels = {
            'ygfB P184Q': 'ygfB P184Q',
            'yfbS V35F': 'yfbS V35F',
            'treB V113E': 'treB V113E',
            'fliI E290D': 'fliI E290D',
            'eutH P393Q': 'eutH P393Q',
            'lrhA/alaA snp_intergenic': 'LrhA-AlaA snp intergenic',
            'lon small_indel': 'Lon indel',
            'hns/tdk mobile_element_insertion': 'hns/Tdk IS insertion',
            'fimE/fimA mobile_element_insertion': 'fimE/fimA IS insertion',
            'dsrA mobile_element_insertion': 'dsrA IS insertion',
            'narQ L21L': 'narQ L21L',
            'rpoS I128N': 'rpoS I128N',
            'pitA A170V': 'pitA A170V',
            'obgE A74T': 'obgE A74T',
            'arcA M39I': 'arcA M39I',
            'yohP/dusC snp_intergenic': 'yohP/dusC intergenic',
            'yhaC/rnpB snp_intergenic': 'yhaC/rnpB intergenic',
            'lysO/aqpZ snp_intergenic': 'lysO/aqpZ intergenic',
            'rpoS small_indel': 'rpoS indel',
            'rcsB mobile_element_insertion': 'rcsB IS insertion',
            'pitA T184T': 'pitA T184T',
            'yaaA I52T': 'yaaA I52T',
            'yhfG/ppiA snp_intergenic': 'yhfG/ppiA intergenic',
            'rlmB/yjfI snp_intergenic': 'rlmB/yjfI intergenic',
            'nrfG/gltP snp_intergenic': 'nrfG/gltP intergenic',
            'manZ/yobD snp_intergenic': 'manZ/yobD intergenic',
            'lon E4E': 'lon E4E',
            'rpoS L125R': 'rpoS L125R',
            'insB9–[crl] large_deletion': 'ΔinsB9-crl'
        }
        manual_label_order = select_mutations
   


    return select_mutations, alt_labels, manual_label_order


In [ ]:
# Assign the group variable
group = 'untreated'  # Options: 'clade', 'PC', 'PL', 'GyrA', 'rpoB', 'PA', 'trkH', 'fusA'

# Select mutations based on group
if group == 'untreated':
    select_mutations = [
        'ygfB P184Q', 'yfbS V35F', 'treB V113E', 'fliI E290D', 'eutH P393Q',
        'lrhA/alaA snp_intergenic', 'lon small_indel', 'hns/tdk mobile_element_insertion',
        'fimE/fimA mobile_element_insertion',
        'dsrA mobile_element_insertion', 'narQ L21L',
        'rpoS I128N', 'pitA A170V', 
        'obgE A74T', 
        'arcA M39I', 'yohP/dusC snp_intergenic',
        'yhaC/rnpB snp_intergenic', 
        'lysO/aqpZ snp_intergenic', 'rpoS small_indel',
        'rcsB mobile_element_insertion', 'pitA T184T', 'yaaA I52T', 'yhfG/ppiA snp_intergenic',
        'rlmB/yjfI snp_intergenic', 'nrfG/gltP snp_intergenic', 'manZ/yobD snp_intergenic',
        'lon E4E', 'rpoS L125R', 'insB9–[crl] large_deletion'
    ]

    alt_labels = {
        'ygfB P184Q': 'ygfB P184Q',
        'yfbS V35F': 'yfbS V35F',
        'treB V113E': 'treB V113E',
        'fliI E290D': 'fliI E290D',
        'eutH P393Q': 'eutH P393Q',
        'lrhA/alaA snp_intergenic': 'LrhA-AlaA snp intergenic',
        'lon small_indel': 'Lon indel',
        'hns/tdk mobile_element_insertion': 'hns/Tdk IS insertion',
        'fimE/fimA mobile_element_insertion': 'fimE/fimA IS insertion',
        'dsrA mobile_element_insertion': 'dsrA IS insertion',
        'narQ L21L': 'narQ L21L',
        'rpoS I128N': 'rpoS I128N',
        'pitA A170V': 'pitA A170V',
        'obgE A74T': 'obgE A74T',
        'arcA M39I': 'arcA M39I',
        'yohP/dusC snp_intergenic': 'yohP/dusC intergenic',
        'yhaC/rnpB snp_intergenic': 'yhaC/rnpB intergenic',
        'lysO/aqpZ snp_intergenic': 'lysO/aqpZ intergenic',
        'rpoS small_indel': 'rpoS indel',
        'rcsB mobile_element_insertion': 'rcsB IS insertion',
        'pitA T184T': 'pitA T184T',
        'yaaA I52T': 'yaaA I52T',
        'yhfG/ppiA snp_intergenic': 'yhfG/ppiA intergenic',
        'rlmB/yjfI snp_intergenic': 'rlmB/yjfI intergenic',
        'nrfG/gltP snp_intergenic': 'nrfG/gltP intergenic',
        'manZ/yobD snp_intergenic': 'manZ/yobD intergenic',
        'lon E4E': 'lon E4E',
        'rpoS L125R': 'rpoS L125R',
        'insB9–[crl] large_deletion': 'ΔinsB9-crl'
    }

    palet = sns.color_palette("muted", n_colors=len(select_mutations))
    select_mut_colors = {val: palet[key] for key, val in enumerate(select_mutations)}


In [ ]:
# af = [traceAlleleFreq(pop, min_freq=0.1) for pop in pops]


In [ ]:
# Build a mapping from population name (like 'P_01') to its sorted timepoints
pop_to_days = {
    pop: meta_subset.query(f"population == '{pop}'").sort_values('Day')['Day'].tolist()
    for pop in pops
}


all = []
for i, (pop, df) in enumerate(zip(pops, af)):  # pop = 'P_01', i = 0
    days = pop_to_days[pop]
    for _, row in df.iterrows():
        for j, freq in enumerate(row['freq']):
            all.append({
                'Culture': i + 1,
                'Day': days[j],  # ✅ Correct day value
                'label': row['label'],
                'gene_name': row['gene_name'],
                'freq': freq
            })
flat_df = pd.DataFrame(all)
flat_df


In [ ]:
flat_df['Culture_Day'] = flat_df['Culture'].astype(str) + "_Day" + flat_df['Day'].astype(str)
flat_df

In [ ]:


# Step 1: Ensure Culture_Day column is in expected format: e.g., '1_Day2', '3_Day4'
# Keep only well-formatted entries
flat_df = flat_df[flat_df['Culture_Day'].str.contains(r'^\d+_Day\d+$', regex=True)]

# Optional: Report dropped malformed entries
# dropped = flat_df[~flat_df['Culture_Day'].str.contains(r'^\d+_Day\d+$', regex=True)]
# print("Dropped malformed entries:", dropped)

# Step 2: Create pivot table with freq values
heatmap_data = flat_df.pivot_table(
    index='label',
    columns='Culture_Day',
    values='freq',
    aggfunc='mean',     # or 'sum', depending on your use case
    fill_value=0        # ensure no NaNs
)

# Step 3: Sort columns by (day, culture)
col_day_culture_pairs = [
    (int(col.split('Day')[1]), int(col.split('_')[0]), col) for col in heatmap_data.columns
]

sorted_cols = [col for _, _, col in sorted(col_day_culture_pairs)]

# Step 4: Reorder columns in the heatmap
heatmap_data = heatmap_data[sorted_cols]

# Now heatmap_data is clean, filled, and ready for plotting


heatmap_data


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
import numpy as np
import matplotlib.gridspec as gridspec
plt.rcParams["font.family"] = "Nimbus Roman"
# --- Step 1: Manually define alt_labels here (replace with your edits) ---
alt_labels = {
    'hns/tdk mobile_element_insertion': r'$\it{hns/tdk}$ IS insertion',
    'fimE/fimA mobile_element_insertion': r'$\it{fimE/fimA}$ IS insertion',
    'fliI E290D': 'FliI E290D',
    'yfbS V35F': 'YfbS V35F',
    'yhaC/rnpB snp_intergenic': r'$\it{yhaC/rnpB}$ snp intergenic',
    'rpoS I128N': 'RpoS I128N',
    'rpoS small_indel': r'$\it{rpoS}$ indel',
    'rpoS L125R': 'RpoS L125R',
    'yhfG/ppiA snp_intergenic': r'$\it{yhfG/ppiA}$ snp intergenic',
    'pitA T184T': 'PitA T184T',
    'pitA A170V': 'PitA A170V',
    'rlmB/yjfI snp_intergenic': r'$\it{rlmB/yjfI}$ snp intergenic',
    'treB V113E': 'TreB V113E',
    'lrhA/alaA snp_intergenic': r'$\it{lrhA/alaA}$ snp intergenic',
    'lon small_indel': r'$\it{lon}$ indel',
    'dsrA mobile_element_insertion': r'$\it{dsrA}$ IS insertion',
    'rcsB mobile_element_insertion': r'$\it{rcsB}$ IS insertion',
    'obgE A74T': 'ObgE A74T',
    'arcA M39I': 'ArcA M39I',
    'ygfB P184Q': 'YgfB P184Q',
    'lon E4E': 'Lon E4E',
    'insB9–[crl] large_deletion': r'$\Delta$insB9-crl',
    'eutH P393Q': 'EutH P393Q',
    'narQ L21L': 'NarQ L21L',
    'yohP/dusC snp_intergenic': r'$\it{yohP/dusC}$ snp intergenic',
    'lysO/aqpZ snp_intergenic': r'$\it{lysO/aqpZ}$ snp intergenic',
    'nrfG/gltP snp_intergenic': r'$\it{nrfG/gltP}$ snp intergenic',
    'yaaA I52T': 'YaaA I52T',
    'manZ/yobD snp_intergenic': r'$\it{manZ/yobD}$ snp intergenic',
}

# --- Step 2: Apply manual row order ---
manual_gene_order = list(alt_labels.keys())
heatmap_data = heatmap_data.reindex(manual_gene_order)

# --- Step 3: Custom colormap (white = 0) ---
standard_map = plt.cm.get_cmap('Oranges')
new_colors = standard_map(np.linspace(0, 1, 256))
new_colors[0] = np.array([1, 1, 1, 1])
custom_map = mcolors.ListedColormap(new_colors)

# --- Step 4: Figure setup ---
fig = plt.figure(figsize=(10, 16))
gs = gridspec.GridSpec(1, 2, width_ratios=[20, 0.6], wspace=0.03)
ax = fig.add_subplot(gs[0])
cbar_ax = fig.add_subplot(gs[1])

# --- Step 5: Plot heatmap ---
sns.heatmap(
    heatmap_data,
    cmap=custom_map,
    vmin=0,
    vmax=1,
    ax=ax,
    cbar_ax=cbar_ax,
    linewidths=0.5,
    linecolor='lightgrey',
    yticklabels=True,
    xticklabels=True,
    cbar_kws={'shrink': 0.8, 'aspect': 10}
)

# --- Step 6: Format x-axis labels ---
xtick_labels = [col.split('_')[0] for col in heatmap_data.columns]
ax.set_xticklabels(xtick_labels, rotation=0, fontsize=12)
ax.set_xlabel("Culture and Day", fontsize=14, labelpad=19)
cbar_ax.tick_params(labelsize=14)

# --- Step 7: Apply alternate labels to y-axis ---
ytick_labels = [tick.get_text() for tick in ax.get_yticklabels()]
alt_ytick_labels = [alt_labels.get(label, label) for label in ytick_labels]
ax.set_yticklabels(alt_ytick_labels, fontsize=14)

# --- Step 8: Add day group labels ---
col_days = [int(col.split('Day')[1]) for col in heatmap_data.columns]
unique_days = sorted(set(col_days))
num_cultures = 4
for i, day in enumerate(unique_days):
    xpos = i * num_cultures + (num_cultures / 2)
    ax.text(
        xpos, -0.024,
        f"Day {day}",
        ha='center', va='top',
        fontsize=14,
        transform=ax.get_xaxis_transform()
    )

# --- Step 9: Final styling and save ---
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(2)
    spine.set_color("black")

plt.subplots_adjust(bottom=0.1)

x_positions = [4, 8, 12]
line_length = heatmap_data.shape[0]
for x in x_positions:
    ax.vlines(x=x, ymin=0, ymax=line_length, colors='black', linestyles='solid', linewidth=1)

plt.tight_layout()
plt.show()

# --- Step 10: Save figure ---
fig.savefig(
    "data/genomics/figures/P_heatmap.png",
    dpi=600,
    bbox_inches='tight'
)
